# 🧠 Thesis Tracking — Demo Notebook

A **thin presentation layer** over `stockanalysis.thesis` — the lifecycle logic
lives in the package (`src/stockanalysis/thesis/`); here we just call it and render.

It records *why* a position was opened, when to review it, and how it turned out,
closing the **Plan → Trade → Record → Review → Improve** loop the signal engine alone can't.

Run **all cells top to bottom**. The lifecycle cells are **fully offline** (state goes to a
temp dir). Only the final *MAE/MFE* cell needs network (Yahoo Finance via `yfinance`).

## 0 · Setup

Make the package importable and point thesis state at a throwaway temp directory so
re-running the demo never touches your real `data/theses/`.

In [ ]:
import sys, tempfile
sys.path.insert(0, "../src")          # run without installing; or: pip install -e ..

import pandas as pd
from stockanalysis.thesis import sources, store, review

STATE = tempfile.mkdtemp(prefix="thesis_demo_")   # isolated, disposable
print("thesis state dir:", STATE)

## 1 · Ingest from a `signal_matrix`

In real use you'd pass a pipeline run straight in:
`sources.from_signal_matrix(STATE, sa.run().signal_matrix)`.
Here we hand-build a tiny matrix so the demo stays offline. Only **Buy** rows are taken
by default; each becomes an `IDEA` thesis (idempotent — re-running won't duplicate).

In [ ]:
demo_signals = pd.DataFrame({
    "Ticker": ["AAPL", "KO", "XYZ"],
    "Sector": ["Technology", "Consumer Staples", "Energy"],
    "Fundamental Score": [6, 5, 2],
    "Technical Posture": ["Bullish", "Neutral", "Bearish"],
    "Tech Score": [6, 4, 1],
    "Composite": [0.78, 0.61, 0.23],
    "Final Action Signal": ["Buy", "Buy", "Watch"],
})
ids = sources.from_signal_matrix(STATE, demo_signals, actions=("Buy",))
print("registered IDEA theses:", ids)

## 2 · Register a manual idea

An idea you formed yourself. Planning fields (target / stop) map onto the thesis;
anything extra (e.g. a planned share size) is preserved as provenance.

In [ ]:
manual_id = sources.from_manual(STATE, {
    "ticker": "MSFT", "thesis_type": "long_term_value",
    "thesis_statement": "Durable franchise, reasonable price after a pullback.",
    "entry_date": "2026-01-05", "target_price": 380.0, "stop_price": 340.0,
    "shares": 8,
})
print("manual thesis:", manual_id)

## 3 · Walk one thesis through its lifecycle

`IDEA → ENTRY_READY → ACTIVE → CLOSED`. P&L is ledger-based and computed on close.

In [ ]:
tid = manual_id
store.transition(STATE, tid, "ENTRY_READY", reason="setup confirmed", event_date="2026-01-05")
store.open_position(STATE, tid, actual_price=352.0, actual_date="2026-01-06", shares=8)
closed = store.close(STATE, tid, exit_reason="target_hit", actual_price=395.0,
                     actual_date="2026-05-20")
print(closed["status"], "| P&L $", closed["outcome"]["pnl_dollars"],
      "(", closed["outcome"]["pnl_pct"], "%) over",
      closed["outcome"]["holding_days"], "days")

## 4 · The thesis ledger (styled)

All presentation stays here in the notebook — the package returns plain dicts.

In [ ]:
def ledger(state):
    rows = [{
        "Ticker": t["ticker"], "Type": t["thesis_type"], "Status": t["status"],
        "Entry": t["entry"]["actual_price"], "P&L %": t["outcome"]["pnl_pct"],
        "Next review": t["monitoring"]["next_review_date"],
    } for t in store.query(state)]
    return pd.DataFrame(rows)

df = ledger(STATE)
(df.style.format({"P&L %": "{:+.1f}%"}, na_rep="—")
   .set_caption("Thesis ledger"))

## 5 · Postmortem & summary

`generate_postmortem` renders a markdown report for a closed thesis. Offline it omits
MAE/MFE; pass a `YFinancePriceAdapter` (next cell) to fill them in from live prices.

In [ ]:
from IPython.display import Markdown

path = review.generate_postmortem(STATE, tid)        # offline: no MAE/MFE
Markdown(open(path).read())

In [ ]:
import json
print(json.dumps(review.summary_stats(STATE), indent=2))

## 7 · Aggregated HTML report

`report.write_report` saves a timestamped journal of **every** thesis to
`output/theses/<timestamp>/report.html`. The same page renders inline below via
the pure `report.build_html_report` renderer (no network).

In [ ]:
from IPython.display import HTML
from stockanalysis.thesis import report

# Notebook runs from notebooks/, so write one level up into the repo's output/.
path = report.write_report(STATE, out_base="../output/theses")
print("Report written:", path)

# Render the same report inline.
HTML(report.build_html_report(store.query(STATE),
                              review.summary_stats(STATE),
                              generated_at="(demo)"))

## 6 · MAE / MFE from live prices  *(needs network)*

Recomputes the postmortem with max adverse / favorable excursion pulled from Yahoo
Finance — the same `yfinance` source the rest of the package uses (no paid API key).

In [ ]:
# Requires network. Comment out to keep the notebook fully offline.
path = review.generate_postmortem(STATE, tid, price_adapter=review.YFinancePriceAdapter())
from IPython.display import Markdown
Markdown(open(path).read())